# Phase 1 — NCAA MBB Data Pull & Validation

**Goal:** Pull one season (2024) of play-by-play, team box score, and schedule data from the
[sportsdataverse](https://github.com/sportsdataverse) parquet CDN (ESPN-derived data).
Validate columns and a few rows before scaling to 2015–2024.

---

## 0 — Setup & Config

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import os, warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', 200)

SEASON = 2024

# sportsdataverse hosts pre-built parquet files on GitHub releases
BASE_URL = "https://github.com/sportsdataverse/sportsdataverse-data/releases/download"

PBP_URL       = f"{BASE_URL}/espn_mens_college_basketball_pbp/play_by_play_{SEASON}.parquet"
TEAM_BOX_URL  = f"{BASE_URL}/espn_mens_college_basketball_team_boxscores/team_box_{SEASON}.parquet"
# Corrected filename for MBB schedule
SCHEDULE_URL  = f"{BASE_URL}/espn_mens_college_basketball_schedules/mbb_schedule_{SEASON}.parquet"

DATA_DIR = os.path.join(os.getcwd(), "data")
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Season: {SEASON}")
print(f"Data dir: {DATA_DIR}")
print(f"pandas {pd.__version__}, pyarrow {pa.__version__}")

## Helper — Download parquet if not cached

In [ ]:
import urllib.request

def download_parquet(url, filename):
    """Download a parquet file from URL, cache locally, return as DataFrame."""
    filepath = os.path.join(DATA_DIR, filename)
    if not os.path.exists(filepath):
        print(f"Downloading from {url} ...")
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"  -> saved to {filepath}")
        except Exception as e:
            print(f"  FAILED to download {filename}: {e}")
            return pd.DataFrame()  # Return empty for safety
    else:
        print(f"Using cached {filename}")
    
    # Bypass the pandas.read_parquet wrapper which often crashes due to extension type conflicts.
    # Loading via pyarrow.parquet directly is much more stable in mixed-version environments.
    try:
        table = pq.read_table(filepath)
        df = table.to_pandas()
        print(f"  Shape: {df.shape[0]:,} rows x {df.shape[1]} cols")
        return df
    except Exception as e:
        print(f"  Failed to read {filename} with pyarrow: {e}")
        print("  Attempting pandas fallback...")
        return pd.read_parquet(filepath)

---
## 1 — Play-by-Play Data

In [ ]:
pbp = download_parquet(PBP_URL, f"pbp_{SEASON}.parquet")

In [ ]:
if not pbp.empty:
    print("=== PBP Columns ===")
    for i, col in enumerate(pbp.columns):
        print(f"  {i:3d}  {col}")
else:
    print("PBP data is empty - check download step")

In [ ]:
if not pbp.empty:
    # Critical columns for the endgame analysis (from task.txt)
    REQUIRED_PBP_COLS = [
        'game_id',
        'period_number',
        'clock_display_value',
        'home_score',
        'away_score',
        'scoring_play',
        'score_value',
        'type_text',
        'text',
    ]

    print("=== Critical Column Check ===")
    for col in REQUIRED_PBP_COLS:
        present = col in pbp.columns
        status = "OK" if present else "MISSING"
        print(f"  {status:>7}  {col}")

    missing = [c for c in REQUIRED_PBP_COLS if c not in pbp.columns]
    if missing:
        print(f"\n  Missing columns: {missing}")
        print("Available columns that look similar:")
        for m in missing:
            candidates = [c for c in pbp.columns if m.split('_')[0] in c.lower()]
            print(f"  {m} -> {candidates}")
    else:
        print("\n  All critical PBP columns present!")

In [ ]:
if not pbp.empty:
    # Show sample rows - pick one game and display event flow
    sample_game_id = pbp['game_id'].iloc[0]
    sample_game = pbp[pbp['game_id'] == sample_game_id]

    # Select the columns that exist from our required list, plus a few extras
    show_cols = [c for c in REQUIRED_PBP_COLS if c in pbp.columns]
    # Also try to include team-related columns
    team_cols = [c for c in pbp.columns if 'team' in c.lower() and 'id' in c.lower()]
    show_cols = show_cols + [c for c in team_cols if c not in show_cols]

    print(f"\n=== Sample Game: {sample_game_id} ({len(sample_game)} events) ===")
    print(f"Showing first 10 rows with key columns:")
    display(sample_game[show_cols].head(10))

In [ ]:
if not pbp.empty:
    # Check the last 10 events of the same game (endgame behavior)
    print(f"=== Last 10 events of game {sample_game_id} (2nd half) ===")
    half2 = sample_game[sample_game['period_number'] == 2] if 'period_number' in sample_game.columns else sample_game
    display(half2[show_cols].tail(10))

---
## 2 — Team Box Score Data

In [ ]:
team_box = download_parquet(TEAM_BOX_URL, f"team_box_{SEASON}.parquet")

In [ ]:
if not team_box.empty:
    print("=== Team Box Score Columns ===")
    for i, col in enumerate(team_box.columns):
        print(f"  {i:3d}  {col}")

In [ ]:
if not team_box.empty:
    # Columns needed for defensive efficiency:
    # FGA, offensive rebounds (OR), turnovers (TO), FTA -> possessions estimate
    # Points allowed -> defensive efficiency
    DEF_EFFICIENCY_COLS = [
        'game_id', 'team_id', 'team_display_name',
        'field_goals_attempted', 'offensive_rebounds',
        'turnovers', 'free_throws_attempted',
        'team_score', 'opponent_team_score',
    ]

    # Check which exist
    print("=== Defensive Efficiency Column Check ===")
    for col in DEF_EFFICIENCY_COLS:
        present = col in team_box.columns
        status = "OK" if present else "MISSING"
        print(f"  {status:>7}  {col}")

    missing_box = [c for c in DEF_EFFICIENCY_COLS if c not in team_box.columns]
    if missing_box:
        print(f"\n  Missing: {missing_box}")
        print("\nSearching for similar columns...")
        for m in missing_box:
            key = m.split('_')[0]
            candidates = [c for c in team_box.columns if key in c.lower()]
            print(f"  {m} -> {candidates}")

In [ ]:
if not team_box.empty:
    # Sample rows
    print(f"=== Sample Team Box Rows (first 6) ===")
    available_cols = [c for c in DEF_EFFICIENCY_COLS if c in team_box.columns]
    display(team_box[available_cols].head(6))

---
## 3 — Schedule Data

In [ ]:
schedule = download_parquet(SCHEDULE_URL, f"schedule_{SEASON}.parquet")

In [ ]:
if not schedule.empty:
    print("=== Schedule Columns ===")
    for i, col in enumerate(schedule.columns):
        print(f"  {i:3d}  {col}")

In [ ]:
if not schedule.empty:
    # Check for home/away/neutral, conference info
    SCHEDULE_COLS = [
        'game_id',
        'home_id', 'away_id',
        'home_display_name', 'away_display_name',
        'conference_id', 'groups_name',
        'season',
        'neutral_site',
        'season_type',
    ]

    print("=== Schedule Column Check ===")
    for col in SCHEDULE_COLS:
        present = col in schedule.columns
        status = "OK" if present else "MISSING"
        print(f"  {status:>7}  {col}")

    missing_sched = [c for c in SCHEDULE_COLS if c not in schedule.columns]
    if missing_sched:
        print(f"\n  Missing: {missing_sched}")
        print("\nSearching for similar columns...")
        for m in missing_sched:
            key = m.split('_')[0]
            candidates = [c for c in schedule.columns if key in c.lower()]
            print(f"  {m} -> {candidates}")

In [ ]:
if not schedule.empty:
    # Sample schedule rows
    available_sched = [c for c in SCHEDULE_COLS if c in schedule.columns]
    print(f"=== Sample Schedule Rows ===")
    display(schedule[available_sched].head(6))

---
## 4 — Null & Coverage Report

In [ ]:
def null_report(df, name, key_cols):
    """Print null counts and percentages for key columns."""
    if df.empty:
        print(f"\n{name} is empty -- skipping null report")
        return
    existing = [c for c in key_cols if c in df.columns]
    total = len(df)
    print(f"\n{'='*50}")
    print(f"{name}  --  {total:,} rows")
    print(f"{'='*50}")
    print(f"{'Column':<35} {'Nulls':>8} {'%':>8}")
    print(f"{'-'*35} {'-'*8} {'-'*8}")
    for col in existing:
        nulls = df[col].isna().sum()
        pct = 100 * nulls / total if total > 0 else 0
        flag = ' WARNING' if pct > 5 else ''
        print(f"{col:<35} {nulls:>8,} {pct:>7.1f}%{flag}")

null_report(pbp, "Play-by-Play", REQUIRED_PBP_COLS if not pbp.empty else [])
null_report(team_box, "Team Box Scores", DEF_EFFICIENCY_COLS if not team_box.empty else [])
null_report(schedule, "Schedule", SCHEDULE_COLS if not schedule.empty else [])

In [ ]:
# Coverage check: how many unique games?
pbp_games = pbp['game_id'].nunique() if not pbp.empty else 0
box_games = team_box['game_id'].nunique() if not team_box.empty and 'game_id' in team_box.columns else 0
sched_games = schedule['game_id'].nunique() if not schedule.empty and 'game_id' in schedule.columns else 0

print(f"\n=== Game Coverage ({SEASON}) ===")
print(f"  PBP unique games:       {pbp_games:,}")
print(f"  Box score unique games: {box_games:,}")
print(f"  Schedule unique games:  {sched_games:,}")

if pbp_games > 3000:
    print("  PBP game count looks reasonable")
elif not pbp.empty:
    print("  WARNING: PBP game count seems low -- investigate")

---
## 5 — Quick Sanity: Clock Monotonicity & Score Progression

In [ ]:
# Check if clock_display_value can be parsed to seconds
def clock_to_seconds(clock_str):
    """Convert 'MM:SS' clock display to seconds remaining."""
    try:
        if pd.isna(clock_str):
            return None
        parts = str(clock_str).split(':')
        if len(parts) == 2:
            return int(parts[0]) * 60 + int(parts[1])
        return None
    except:
        return None

# Test on sample game
if not pbp.empty and 'clock_display_value' in pbp.columns:
    sample = pbp[pbp['game_id'] == sample_game_id].copy()
    sample['secs_remaining'] = sample['clock_display_value'].apply(clock_to_seconds)
    
    parseable = sample['secs_remaining'].notna().sum()
    total_events = len(sample)
    print(f"Clock parsing success: {parseable}/{total_events} events ({100*parseable/total_events:.1f}%)")
    
    # Show the last few events of 2nd half with parsed clock
    half2_cols = ['period_number', 'clock_display_value', 'home_score', 'away_score', 'type_text', 'text']
    half2_cols = [c for c in half2_cols if c in sample.columns]
    h2 = sample[sample['period_number'] == 2] if 'period_number' in sample.columns else sample
    print(f"\nLast 10 events of 2nd half (game {sample_game_id}):")
    display(h2[half2_cols + ['secs_remaining']].tail(10))
elif not pbp.empty:
    print("WARNING: clock_display_value not found -- check column listing above")

In [ ]:
# Check score monotonicity -- scores should only increase
if not pbp.empty and 'home_score' in pbp.columns and 'away_score' in pbp.columns:
    sample = pbp[pbp['game_id'] == sample_game_id].copy()
    sample['home_score'] = pd.to_numeric(sample['home_score'], errors='coerce')
    sample['away_score'] = pd.to_numeric(sample['away_score'], errors='coerce')
    
    home_decreases = (sample['home_score'].diff() < 0).sum()
    away_decreases = (sample['away_score'].diff() < 0).sum()
    
    # Period breaks may cause score 'resets' in display -- check within period
    print(f"Score decrease events (home): {home_decreases}")
    print(f"Score decrease events (away): {away_decreases}")
    if home_decreases <= 1 and away_decreases <= 1:
        print("Score progression looks monotonic (<=1 decrease, likely period boundary)")
    else:
        print("WARNING: Multiple score decreases detected -- investigate data ordering")
elif not pbp.empty:
    print("WARNING: home_score / away_score columns not found")

---
## 6 — Summary & Next Steps

**If all checks pass above**, the data supports the endgame scoring analysis in `task.txt`.

**Next steps (Phase 2):**
1. Scale to 2015-2024: `for season in range(2015, 2025): download_parquet(...)`
2. Parse clock -> seconds remaining in regulation
3. Build margin buckets and exposure table
4. Compute team-season defensive efficiency from box scores
5. Tag foul/FT events for endgame regime detection

In [ ]:
pbp_rows = pbp.shape[0] if not pbp.empty else 0
pbp_cols = pbp.shape[1] if not pbp.empty else 0
box_rows = team_box.shape[0] if not team_box.empty else 0
box_cols = team_box.shape[1] if not team_box.empty else 0
sched_rows = schedule.shape[0] if not schedule.empty else 0
sched_cols = schedule.shape[1] if not schedule.empty else 0

print("Phase 1 data pull complete.")
print(f"  PBP:      {pbp_rows:>10,} rows, {pbp_cols:>3} cols, {pbp_games:,} games")
print(f"  Box:      {box_rows:>10,} rows, {box_cols:>3} cols, {box_games:,} games")
print(f"  Schedule: {sched_rows:>10,} rows, {sched_cols:>3} cols, {sched_games:,} games")